# MarketMind — Offline GMM Training & Chronological Evaluation

This notebook trains a **Gaussian Mixture Model (GMM)** on real NIFTY 50 market data, evaluates it on a chronological holdout set, and saves the trained model plus a JSON training report.

**Input:** `data/processed/nifty50_yahoo_3m.csv`  
**Model output:** `ml/models/nifty50_gmm_3m.joblib`  
**Evaluation:** Unsupervised clustering metrics on the final 25% of observations.

> The clustering metrics do not measure directional prediction accuracy or trading profitability.


## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

from indicators.technical_indicators import compute_all
from ml.feature_engineering import build_features
from ml.model_utils import REGIMES, assign_regime_labels, describe_clusters
from ml.regime_detection import LOG_COLUMNS, _prep


# Update these paths if your project structure is different.
INPUT_CSV = Path("data/processed/nifty50_yahoo_3m.csv")
OUTPUT_MODEL = Path("ml/models/nifty50_gmm_3m.joblib")

TEST_FRACTION = 0.25
MIN_ROWS = 250
RANDOM_SEED = 42

REAL_FEATURE_COLUMNS = [
    "ret_5",
    "roll_vol",
    "rsi",
    "ma_diff",
    "momentum",
    "atr_pct",
]


## 2. Load and validate candle data

In [ ]:
def load_candles(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["timestamp"])
    df = df.set_index("timestamp").sort_index()

    rename = {c: c.lower() for c in df.columns}
    df = df.rename(columns=rename)

    required = ["open", "high", "low", "close"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Dataset missing OHLC columns: {missing}")

    # Index volume is not a meaningful traded-volume measure; it is intentionally not used.
    if "volume" not in df:
        df["volume"] = 0.0

    for c in required:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=required)
    df = df[
        (df["open"] > 0)
        & (df["high"] > 0)
        & (df["low"] > 0)
        & (df["close"] > 0)
    ]
    df = df[~df.index.duplicated(keep="last")]

    return df


candles = load_candles(INPUT_CSV)
print(f"Loaded {len(candles):,} valid candles")
display(candles.head())
display(candles.tail())


## 3. Compute indicators and engineer features

In [ ]:
indicators = compute_all(candles)

features = build_features(indicators, include_volume=False)[REAL_FEATURE_COLUMNS]
features = features.replace([np.inf, -np.inf], np.nan).dropna()

print(f"Complete feature rows: {len(features):,}")
print("Features:", REAL_FEATURE_COLUMNS)
display(features.head())


## 4. Chronological train-test split and preprocessing

In [ ]:
if len(features) < MIN_ROWS:
    raise ValueError(
        f"Only {len(features)} complete feature rows are available; need at least {MIN_ROWS}. "
        "Yahoo's short 1-minute history may be insufficient. Obtain more genuine "
        "1-minute data or use a longer supported timeframe."
    )

split = int(len(features) * (1.0 - TEST_FRACTION))
if split < 150 or len(features) - split < 30:
    raise ValueError("Chronological split leaves too few training or test rows")

X_train = features.iloc[:split]
X_test = features.iloc[split:]

# Volatility features are log-transformed before scaling, matching live inference.
train_prep = _prep(X_train, REAL_FEATURE_COLUMNS)
test_prep = _prep(X_test, REAL_FEATURE_COLUMNS)

scaler = StandardScaler().fit(train_prep)
Z_train = scaler.transform(train_prep)
Z_test = scaler.transform(test_prep)

print(f"Training rows: {len(X_train):,}")
print(f"Testing rows:  {len(X_test):,}")
print(f"Train period:  {X_train.index.min()} to {X_train.index.max()}")
print(f"Test period:   {X_test.index.min()} to {X_test.index.max()}")


## 5. Train the Gaussian Mixture Model

In [ ]:
gmm = GaussianMixture(
    n_components=4,
    covariance_type="full",
    n_init=5,
    random_state=RANDOM_SEED,
    reg_covar=1e-5,
).fit(Z_train)

mapping = assign_regime_labels(gmm.means_, REAL_FEATURE_COLUMNS)

# Convert cluster centers back to the original feature scale.
centers_scaled = gmm.means_
centers_transformed = scaler.inverse_transform(centers_scaled)
centers_raw = centers_transformed.copy()

for col in LOG_COLUMNS:
    if col in REAL_FEATURE_COLUMNS:
        idx = REAL_FEATURE_COLUMNS.index(col)
        centers_raw[:, idx] = np.exp(centers_raw[:, idx])

notes = describe_clusters(mapping, centers_raw, REAL_FEATURE_COLUMNS)

print("Cluster-to-regime mapping:")
display(mapping)
print("Cluster descriptions:")
display(notes)


## 6. Evaluate on the chronological holdout set

In [ ]:
clusters = gmm.predict(Z_test)
unique = np.unique(clusters)

metrics = {
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "feature_rows": int(len(features)),
    "n_clusters_present_in_test": int(len(unique)),
    "silhouette": (
        float(silhouette_score(Z_test, clusters))
        if 1 < len(unique) < len(Z_test)
        else None
    ),
    "davies_bouldin": (
        float(davies_bouldin_score(Z_test, clusters))
        if 1 < len(unique)
        else None
    ),
    "calinski_harabasz": (
        float(calinski_harabasz_score(Z_test, clusters))
        if 1 < len(unique)
        else None
    ),
    "test_cluster_counts": {
        str(int(k)): int((clusters == k).sum()) for k in unique
    },
    "test_regime_counts": {
        mapping[int(k)]: int((clusters == k).sum()) for k in unique
    },
    "evaluation_note": (
        "Unsupervised clustering metrics on a chronological holdout; "
        "these are not directional accuracy or profitability metrics."
    ),
}

print(json.dumps(metrics, indent=2))


## 7. Save the trained model and training report

In [ ]:
limitations = [
    "Yahoo 1-minute data has a short lookback, so this model is trained on a small recent sample.",
    "NIFTY 50 index volume is not used as a feature.",
    "Regime names are assigned from cluster-centroid heuristics; GMM itself is unsupervised.",
    "Clustering metrics do not establish trading profitability or predictive accuracy.",
]

artifact = {
    "model_type": "GaussianMixture",
    "provider": "Yahoo Finance",
    "symbol": "^NSEI",
    "interval": "3m",
    "feature_columns": REAL_FEATURE_COLUMNS,
    "log_columns": [c for c in LOG_COLUMNS if c in REAL_FEATURE_COLUMNS],
    "scaler": scaler,
    "gmm": gmm,
    "mapping": mapping,
    "cluster_notes": notes,
    "training_start": X_train.index.min().isoformat(),
    "training_end": X_train.index.max().isoformat(),
    "test_start": X_test.index.min().isoformat(),
    "test_end": X_test.index.max().isoformat(),
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "metrics": metrics,
    "limitations": limitations,
}

OUTPUT_MODEL.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, OUTPUT_MODEL)

report = {
    "model": artifact["model_type"],
    "source": f'{artifact["provider"]} {artifact["symbol"]}',
    "interval": artifact["interval"],
    "features": REAL_FEATURE_COLUMNS,
    "train_period": [artifact["training_start"], artifact["training_end"]],
    "test_period": [artifact["test_start"], artifact["test_end"]],
    "cluster_mapping": {str(k): v for k, v in mapping.items()},
    "cluster_notes": notes,
    "metrics": metrics,
    "limitations": limitations,
    "artifact": str(OUTPUT_MODEL),
}

report_path = OUTPUT_MODEL.parent / "training_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print(f"Model saved to:  {OUTPUT_MODEL}")
print(f"Report saved to: {report_path}")
display(report)


## 8. Notes for interpretation

- The split is chronological: earlier observations train the model and later observations evaluate it.
- The scaler is fitted only on the training data, avoiding leakage from the holdout period.
- GMM is unsupervised. Cluster IDs are mapped to human-readable regime labels using centroid heuristics.
- Silhouette is generally better when higher; Davies–Bouldin is generally better when lower; Calinski–Harabasz is generally better when higher.
- These metrics describe cluster separation/compactness. They do **not** demonstrate that the regimes predict future market direction or produce profitable trades.
